# ⚡ Smart Electricity Theft Detection System

> **End-to-end ML project using UCI Electricity Load Diagrams 2011–2014**

---

### Notebook Contents
1. Setup & Imports
2. Data Loading & Exploration
3. Data Preprocessing
4. Feature Engineering
5. Model Training
   - Isolation Forest
   - Local Outlier Factor
   - Random Forest
6. Evaluation
7. Visualisations
8. Sample Predictions


## 1. Setup & Imports

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.neighbors import LocalOutlierFactor
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
import joblib

# Add src to path
sys.path.insert(0, 'src')

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor'  : '#16213e',
    'axes.edgecolor'  : '#334155',
    'axes.labelcolor' : '#e2e8f0',
    'text.color'      : '#e2e8f0',
    'xtick.color'     : '#94a3b8',
    'ytick.color'     : '#94a3b8',
    'grid.color'      : '#334155',
    'grid.alpha'      : 0.4,
    'font.family'     : 'DejaVu Sans',
    'axes.titlesize'  : 14,
    'axes.titleweight': 'bold',
})

print('✅ Imports complete')
print(f'NumPy  : {np.__version__}')
print(f'Pandas : {pd.__version__}')

## 2. Data Loading & Exploration

In [ ]:
# Step 1: Run preprocessing pipeline
from preprocess import run_preprocessing

# Load first 20,000 rows for notebook exploration (full run in run_pipeline.py)
df_scaled = run_preprocessing(nrows=20000)

In [ ]:
print(f'Shape       : {df_scaled.shape}')
print(f'Date range  : {df_scaled.index.min()} → {df_scaled.index.max()}')
print(f'Clients     : {df_scaled.shape[1]}')
print(f'\nFirst 5 rows:')
df_scaled.iloc[:5, :6]

In [ ]:
# Plot first 5 clients — raw consumption
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
for i, (col, ax) in enumerate(zip(df_scaled.columns[:5], axes)):
    ax.plot(df_scaled.index, df_scaled[col], linewidth=0.8, color='#64b3f4', alpha=0.9)
    ax.set_ylabel(col, fontsize=9)
    ax.grid(True)
axes[0].set_title('Normalised Hourly Consumption — First 5 Clients', fontsize=14, fontweight='bold', color='white')
axes[-1].set_xlabel('Date-Time', color='white')
plt.tight_layout()
plt.savefig('screenshots/notebook_5clients.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Missing values check
missing = df_scaled.isna().sum().sum()
print(f'Missing values after preprocessing: {missing}')

# Basic statistics for first 10 clients
df_scaled.iloc[:, :10].describe().round(4)

## 3. Feature Engineering

In [ ]:
from features import build_feature_matrix

# Build features for first 20 clients
feat_df = build_feature_matrix(df_scaled, sample_clients=20)
feat_df.head()

In [ ]:
# Visualise feature distributions
numeric_feats = ['consumption', 'rolling_mean_24h', 'rolling_std_24h', 'prev_day_diff']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()
colors = ['#64b3f4', '#a78bfa', '#34d399', '#fb923c']
for i, (col, ax, c) in enumerate(zip(numeric_feats, axes, colors)):
    ax.hist(feat_df[col].dropna(), bins=60, color=c, alpha=0.85, edgecolor='none')
    ax.set_title(col.replace('_', ' ').title())
    ax.grid(True)
fig.suptitle('Feature Distributions', fontsize=15, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig('screenshots/notebook_feature_dists.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: avg consumption by hour × day
pivot = feat_df.pivot_table(values='consumption', index='hour', columns='day_of_week', aggfunc='mean')
pivot.columns = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.3, ax=ax,
            cbar_kws={'label': 'Avg Normalised Consumption'})
ax.set_title('Average Consumption: Hour × Day of Week', fontsize=14, fontweight='bold')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Hour of Day')
plt.tight_layout()
plt.savefig('screenshots/notebook_hour_day_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()

## 4. Simulate Theft Labels

In [ ]:
from train import simulate_theft_labels, FEATURE_COLS

y = simulate_theft_labels(feat_df)
X = feat_df[FEATURE_COLS].values

print(f'Feature matrix: {X.shape}')
print(f'Label distribution:')
print(y.value_counts().rename({0:'Normal', 1:'Suspicious'}))

## 5. Model Training

In [ ]:
# ── Isolation Forest ──────────────────────────────────────────────────────────
from train import train_isolation_forest
if_model = train_isolation_forest(X)
if_scores = if_model.score_samples(X)
if_preds_raw = if_model.predict(X)
if_preds = (if_preds_raw == -1).astype(int)   # 1 = suspicious

print(f'IF flagged {if_preds.sum():,} anomalies ({100*if_preds.mean():.1f}%)')

In [ ]:
# ── LOF ───────────────────────────────────────────────────────────────────────
from train import train_lof
lof_model = train_lof(X)
lof_preds_raw = lof_model.predict(X)
lof_preds = (lof_preds_raw == -1).astype(int)

print(f'LOF flagged {lof_preds.sum():,} anomalies ({100*lof_preds.mean():.1f}%)')

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
from train import train_random_forest

X_tr, X_te, y_tr, y_te = train_test_split(X, y.values, test_size=0.2, random_state=42, stratify=y.values)
rf_model = train_random_forest(X_tr, y_tr)
rf_preds  = rf_model.predict(X_te)

print('Random Forest training complete')

## 6. Evaluation

In [ ]:
from sklearn.metrics import classification_report

for name, preds in [('Isolation Forest', if_preds),
                     ('LOF', lof_preds)]:
    print(f'\n── {name} ──')
    print(classification_report(y.values, preds, target_names=['Normal', 'Suspicious']))

print('\n── Random Forest (test set) ──')
print(classification_report(y_te, rf_preds, target_names=['Normal', 'Suspicious']))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, name, preds, y_true, cmap in [
    (axes[0], 'Isolation Forest', if_preds,  y.values, 'Blues'),
    (axes[1], 'LOF',              lof_preds, y.values, 'Purples'),
    (axes[2], 'Random Forest',    rf_preds,  y_te,     'Greens'),
]:
    cm = confusion_matrix(y_true, preds)
    ConfusionMatrixDisplay(cm, display_labels=['Normal','Suspicious']).plot(ax=ax, cmap=cmap)
    ax.set_title(name, fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('screenshots/notebook_confusion_matrices.png', dpi=130, bbox_inches='tight')
plt.show()

## 7. Visualisations

In [ ]:
# Time-series with anomaly highlights
ts_df = feat_df[['datetime','consumption']].copy()
ts_df['label'] = np.where(if_preds == 1, 'Suspicious', 'Normal')
ts_df = ts_df.sort_values('datetime')

# Sample for plot clarity
sample = ts_df.sample(min(2000, len(ts_df)), random_state=42).sort_values('datetime')

fig, ax = plt.subplots(figsize=(14, 5))
normal    = sample[sample['label'] == 'Normal']
suspicious = sample[sample['label'] == 'Suspicious']

ax.scatter(normal['datetime'], normal['consumption'],
           color='#64b3f4', s=6, alpha=0.4, label='Normal')
ax.scatter(suspicious['datetime'], suspicious['consumption'],
           color='#ef4444', s=12, alpha=0.9, zorder=5, label='Suspicious')
ax.set_title('Electricity Consumption — Anomalies Highlighted', fontsize=14, fontweight='bold')
ax.set_xlabel('Date-Time')
ax.set_ylabel('Normalised Consumption')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig('screenshots/notebook_timeseries.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Anomaly score distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(if_scores, bins=80, color='#64b3f4', alpha=0.85, edgecolor='none', label='All points')
threshold = np.percentile(if_scores, 5)
ax.axvline(threshold, color='#ef4444', linestyle='--', linewidth=2,
           label=f'Anomaly threshold (5th percentile = {threshold:.3f})')
ax.set_title('Isolation Forest — Anomaly Score Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Anomaly Score  (lower = more suspicious)')
ax.set_ylabel('Count')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig('screenshots/notebook_anomaly_scores.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
importances  = rf_model.feature_importances_
sorted_idx   = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(11, 5))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(FEATURE_COLS)))
ax.bar(range(len(FEATURE_COLS)), importances[sorted_idx], color=colors, edgecolor='none')
ax.set_xticks(range(len(FEATURE_COLS)))
ax.set_xticklabels([FEATURE_COLS[i] for i in sorted_idx], rotation=40, ha='right')
ax.set_title('Feature Importances — Random Forest', fontsize=14, fontweight='bold')
ax.set_ylabel('Importance')
ax.grid(True, axis='y')
plt.tight_layout()
plt.savefig('screenshots/notebook_feature_importance.png', dpi=130, bbox_inches='tight')
plt.show()

## 8. Sample Predictions

In [ ]:
import random
from predict import predict_single

# ── Test 1: Normal residential profile
normal_profile = [
    1.2, 1.0, 0.9, 0.8, 0.8, 1.0, 1.8, 3.2,
    3.5, 3.0, 2.8, 2.9, 3.0, 2.9, 2.8, 3.1,
    3.8, 4.5, 4.8, 4.5, 3.9, 3.1, 2.2, 1.5
]

# ── Test 2: Suspicious profile (abnormally low — possible meter bypass)
suspicious_profile = [0.05] * 24

for name, profile in [('Normal Residential', normal_profile),
                       ('Suspicious (Meter Bypass)', suspicious_profile)]:
    result = predict_single(profile, model='isolation_forest')
    print(f'\n=== {name} ===')
    print(f'  Prediction : {result["label"]}')
    print(f'  Code       : {result["code"]}  (0=Normal, 1=Suspicious)')
    print(f'  Avg Score  : {result["score"]:.4f}')
    print(f'  Hourly     : {result["hour_labels"][:8]} …')

In [ ]:
# Visualise both profiles side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
hours = list(range(24))

for ax, profile, title, color in [
    (axes[0], normal_profile,    'Normal Profile (Residential)', '#10b981'),
    (axes[1], suspicious_profile, 'Suspicious Profile (Meter Bypass)', '#ef4444'),
]:
    ax.bar(hours, profile, color=color, alpha=0.8, edgecolor='none')
    ax.plot(hours, profile, color='white', linewidth=2, marker='o', markersize=4)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Consumption (kWh)')
    ax.set_xticks(hours)
    ax.grid(True, axis='y')

plt.suptitle('Sample Prediction Profiles', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('screenshots/notebook_sample_profiles.png', dpi=130, bbox_inches='tight')
plt.show()

print('\n✅ Notebook complete! All plots saved to screenshots/')